# 01. Testing the REN DataHub API
First look at the data returned for electricity market prices and production by source.

In [1]:
# Import the libraries
import requests
import pandas as pd

In [2]:
# Ask the REN API for the electricity market prices of one day
url = "https://servicebus.ren.pt/datahubapi/electricity/ElectricityMarketPricesDaily"
params = {"culture": "en-US", "date": "2026-09-30"}
response = requests.get(url, params=params)

# Check if the request worked (200 means OK) and show the data
print(response.status_code)
prices = response.json()
prices

404


{'message': 'No data available'}

In [3]:
# Ask the REN API for the electricity production by source on the same day
url = "https://servicebus.ren.pt/datahubapi/electricity/ElectricityProductionBreakdownDaily"
params = {"culture": "en-US", "date": "2026-09-30"}
response = requests.get(url, params=params)

# Check the request and show the data
print(response.status_code)
production = response.json()
production

200


{'xAxis': {'title': {'text': 'Hour'},
  'categories': ['00:00',
   '00:15',
   '00:30',
   '00:45',
   '01:00',
   '01:15',
   '01:30',
   '01:45',
   '02:00',
   '02:15',
   '02:30',
   '02:45',
   '03:00',
   '03:15',
   '03:30',
   '03:45',
   '04:00',
   '04:15',
   '04:30',
   '04:45',
   '05:00',
   '05:15',
   '05:30',
   '05:45',
   '06:00',
   '06:15',
   '06:30',
   '06:45',
   '07:00',
   '07:15',
   '07:30',
   '07:45',
   '08:00',
   '08:15',
   '08:30',
   '08:45',
   '09:00',
   '09:15',
   '09:30',
   '09:45',
   '10:00',
   '10:15',
   '10:30',
   '10:45',
   '11:00',
   '11:15',
   '11:30',
   '11:45',
   '12:00',
   '12:15',
   '12:30',
   '12:45',
   '13:00',
   '13:15',
   '13:30',
   '13:45',
   '14:00',
   '14:15',
   '14:30',
   '14:45',
   '15:00',
   '15:15',
   '15:30',
   '15:45',
   '16:00',
   '16:15',
   '16:30',
   '16:45',
   '17:00',
   '17:15',
   '17:30',
   '17:45',
   '18:00',
   '18:15',
   '18:30',
   '18:45',
   '19:00',
   '19:15',
   '19:30',


In [4]:
# Test the prices with the date used in the REN documentation example
url = "https://servicebus.ren.pt/datahubapi/electricity/ElectricityMarketPricesDaily"
params = {"culture": "en-US", "date": "2025-01-25"}
response = requests.get(url, params=params)
print(response.status_code)

# Show the main parts of the answer and the name and size of each series
prices = response.json()
print(prices.keys())
for serie in prices["series"]:
    print(serie["name"], len(serie["data"]))

200
dict_keys(['yAxis', 'xAxis', 'series'])
PT 24
ES 24


In [5]:
# Show the main parts of the production answer and the name and size of each series
print(production.keys())
for serie in production["series"]:
    print(serie["name"], len(serie["data"]))

dict_keys(['xAxis', 'yAxis', 'series'])
Consumption + Storage 96
Consumption 96
Pumping 96
Hydro 96
Solar 96
Import 96
Export 96
Wind 96
Natural Gas 96
Other Thermal 96
Biomass 96
Coal 96
Wave 96
Consumption of Batteries 96
Battery Injection 96


In [6]:
# Check which dates have price data available, and how many values each day has
url = "https://servicebus.ren.pt/datahubapi/electricity/ElectricityMarketPricesDaily"
test_dates = ["2025-09-30", "2025-10-01", "2026-01-15", "2026-06-15", "2026-09-15", "2026-09-28", "2026-10-01"]

for day in test_dates:
    response = requests.get(url, params={"culture": "en-US", "date": day})
    if response.status_code == 200:
        data = response.json()
        print(day, "OK,", len(data["series"][0]["data"]), "values")
    else:
        print(day, "no data, code", response.status_code)

2025-09-30 OK, 24 values
2025-10-01 OK, 24 values
2026-01-15 OK, 24 values
2026-06-15 OK, 24 values
2026-09-15 no data, code 404
2026-09-28 no data, code 404
2026-10-01 no data, code 404


In [7]:
# Find the last day with price data, checking every day from June to September 2026
days = pd.date_range("2026-06-15", "2026-09-30")
last_day = None

for day in days:
    day_text = day.strftime("%Y-%m-%d")
    response = requests.get(url, params={"culture": "en-US", "date": day_text})
    if response.status_code == 200:
        last_day = day_text

print("Last day with price data:", last_day)

Last day with price data: 2026-06-29


In [8]:
# Download the OMIE day-ahead prices for one day (the file name uses the date as YYYYMMDD)
day = "20261002"
url = "https://www.omie.es/en/file-download?parents%5B0%5D=marginalpdbc&filename=marginalpdbc_" + day + ".1"
response = requests.get(url)

# Check the request and show the start of the file
print(response.status_code)
print(response.text[:600])

200
MARGINALPDBC;
2026;10;02;1;185.11;185.11;
2026;10;02;2;180.76;180.76;
2026;10;02;3;171.32;171.32;
2026;10;02;4;168.45;168.45;
2026;10;02;5;169;169;
2026;10;02;6;166.71;166.71;
2026;10;02;7;164.72;164.72;
2026;10;02;8;161.45;161.45;
2026;10;02;9;162.95;162.95;
2026;10;02;10;161;161;
2026;10;02;11;161.43;161.43;
2026;10;02;12;161.47;161.47;
2026;10;02;13;160.29;160.29;
2026;10;02;14;159.51;159.51;
2026;10;02;15;161.62;161.62;
2026;10;02;16;161.2;161.2;
2026;10;02;17;162.92;162.92;
2026;10;02;18;163.12;163.12;
2026;10;02;19;162.05;162.05;
2026;10;02;20;166.6;166.6;
2026;10;02


In [9]:
# Turn the OMIE text file into a table, one row per 15-minute period
def read_omie_day(day):
    url = "https://www.omie.es/en/file-download?parents%5B0%5D=marginalpdbc&filename=marginalpdbc_" + day + ".1"
    response = requests.get(url)
    rows = []
    for line in response.text.splitlines():
        parts = line.split(";")
        if len(parts) >= 6 and parts[0].isdigit():
            rows.append({
                "date": parts[0] + "-" + parts[1] + "-" + parts[2],
                "period": int(parts[3]),
                "price_pt": float(parts[4]),
                "price_es": float(parts[5]),
            })
    return pd.DataFrame(rows)

# Test with today and tomorrow
today = read_omie_day("20261002")
print(len(today), "periods today")
tomorrow = read_omie_day("20261003")
print(len(tomorrow), "periods tomorrow")
today.head()

96 periods today
96 periods tomorrow


,date,period,price_pt,price_es
0,2026-10-02,1,185.11,185.11
1,2026-10-02,2,180.76,180.76
2,2026-10-02,3,171.32,171.32
3,2026-10-02,4,168.45,168.45
4,2026-10-02,5,169.00,169.00


In [10]:
# Check how far back the REN production data goes
url = "https://servicebus.ren.pt/datahubapi/electricity/ElectricityProductionBreakdownDaily"
test_dates = ["2015-01-15", "2018-01-15", "2019-01-15", "2021-06-15", "2023-06-15", "2025-06-15", "2026-10-01"]

for day in test_dates:
    response = requests.get(url, params={"culture": "en-US", "date": day})
    if response.status_code == 200:
        data = response.json()
        print(day, "OK,", len(data["series"]), "series,", len(data["series"][0]["data"]), "values")
    else:
        print(day, "no data, code", response.status_code)

2015-01-15 OK, 15 series, 96 values
2018-01-15 OK, 15 series, 96 values
2019-01-15 OK, 15 series, 96 values
2021-06-15 OK, 15 series, 96 values
2023-06-15 OK, 15 series, 96 values
2025-06-15 OK, 15 series, 96 values
2026-10-01 OK, 15 series, 96 values


In [11]:
# Check that old OMIE files (before 15-minute periods) have the same format
old_day = read_omie_day("20190115")
print(len(old_day), "periods on 15 January 2019")
old_day.head()

0 periods on 15 January 2019


""


In [12]:
# See what OMIE returns for an old date
url = "https://www.omie.es/en/file-download?parents%5B0%5D=marginalpdbc&filename=marginalpdbc_20190115.1"
response = requests.get(url)
print(response.status_code)
print(response.text[:300])

404
<!DOCTYPE html>
<html lang="en" dir="ltr">
  <head>
    <meta charset="utf-8" />
<script async src="https://www.googletagmanager.com/gtag/js?id=G-N4CHXT49HS"></script>
<script>window.dataLayer = window.dataLayer || [];function gtag(){dataLayer.push(arguments)};gtag("js", new Date());gtag("set", "dev


In [13]:
# OMIE prices for 15 June 2026, converted from 15-minute periods to hourly averages
omie_day = read_omie_day("20260615")
omie_day["hour"] = (omie_day["period"] - 1) // 4
omie_hourly = omie_day.groupby("hour")["price_pt"].mean()

# REN prices for the same day (the first series is Portugal)
url = "https://servicebus.ren.pt/datahubapi/electricity/ElectricityMarketPricesDaily"
response = requests.get(url, params={"culture": "en-US", "date": "2026-06-15"})
ren_pt = response.json()["series"][0]["data"]

# Put both side by side to compare
comparison = pd.DataFrame({"omie_hourly_avg": omie_hourly.round(2).values, "ren": ren_pt})
comparison.head(10)

,omie_hourly_avg,ren
0,127.92,127.97
1,125.04,125.06
2,120.27,120.28
3,119.83,119.83
4,120.39,120.40
5,125.04,125.06
6,135.34,135.39
7,134.97,135.02
8,115.10,115.28
9,71.28,70.25
